Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo2\\datosNarmax\\24pasos_lstm_pollution.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [ ]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd,e
date,,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,NaN
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,NaN
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,NaN
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,NaN
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 24
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 0])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43765, 12, 7)
Dimensiones de Y: (43765, 1)


In [9]:
print(datosX[0])

[[ 0.31768099 -1.2140229  -1.26852411  0.32968671 -0.38094383 -0.46404777
          nan]
 [ 0.52615226 -1.14430217 -1.26852411  0.32968671 -0.38094383 -0.44657536
          nan]
 [ 0.64684616 -0.86541928 -1.34931411  0.42612698 -0.38094383 -0.42910295
          nan]
 [ 0.88823396 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.39396181
          nan]
 [ 0.41643054 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.3764894
          nan]
 [ 0.09823754 -0.58653639 -1.4301041   0.52256725 -0.38094383 -0.35901698
          nan]
 [ 0.05434885 -0.58653639 -1.4301041   0.61900753 -0.38094383 -0.32387584
          nan]
 [ 0.26282012 -0.58653639 -1.34931411  0.7154478  -0.38094383 -0.2887347
          nan]
 [ 0.21893143 -0.65625711 -1.4301041   0.7154478  -0.38094383 -0.25359356
          nan]
 [ 0.3505975  -0.58653639 -1.34931411  0.81188808 -0.38094383 -0.21845241
          nan]
 [ 0.43837488 -0.58653639 -1.34931411  0.90832835 -0.38094383 -0.15700449
          nan]
 [ 0.57004095 -0.656257

Se dividen nuevamente los conjuntos de datos

In [10]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30635, 12, 7)
Las dimensiones de testX son:  (8797, 12, 7)
Las dimensiones de valX son:  (4333, 12, 7)


In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30635, 1)
Las dimensiones de testY son:  (8797, 1)
Las dimensiones de valY son:  (4333, 1)


Se crean métricas para medir desempeño

In [12]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [13]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=128,
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

49/49 - 22s - 451ms/step - ia: 0.2592 - loss: 2.6974 - mae: 1.4674 - rmse: 1.6379 - smape: 1.5712 - val_ia: 0.2071 - val_loss: 2.0664 - val_mae: 1.3349 - val_rmse: 1.4139 - val_smape: 1.6711

Epoch 2/128                                           

49/49 - 1s - 21ms/step - ia: 0.2501 - loss: 2.2284 - mae: 1.3227 - rmse: 1.4918 - smape: 1.5773 - val_ia: 0.2261 - val_loss: 1.6493 - val_mae: 1.1787 - val_rmse: 1.2636 - val_smape: 1.6542

Epoch 3/128                                           

49/49 - 2s - 31ms/step - ia: 0.2464 - loss: 1.9470 - mae: 1.2161 - rmse: 1.3913 - smape: 1.5759 - val_ia: 0.2391 - val_loss: 1.3357 - val_mae: 1.0462 - val_rmse: 1.1376 - val_smape: 1.6399

Epoch 4/128                                           

49/49 - 1s - 26ms/step - ia: 0.2346 - loss: 1.7208 - mae: 1.1246 - rmse: 1.3061 - smape: 1.5876 - val_ia: 0.2451 - val_loss: 1.1014 - val_mae: 0.9357 - val_rmse: 1.0328 - val_smape: 1.6313

Epoch 5/128   

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                      

385/385 - 59s - 154ms/step - ia: 0.3750 - loss: 0.9263 - mae: 0.7124 - rmse: 0.9317 - smape: 1.2736 - val_ia: 0.2304 - val_loss: 0.5102 - val_mae: 0.5316 - val_rmse: 0.5942 - val_smape: 1.2640

Epoch 2/128                                                                      

385/385 - 12s - 30ms/step - ia: 0.4150 - loss: 0.8744 - mae: 0.6872 - rmse: 0.9071 - smape: 1.2118 - val_ia: 0.2299 - val_loss: 0.5058 - val_mae: 0.5374 - val_rmse: 0.6004 - val_smape: 1.3024

Epoch 3/128                                                                      

385/385 - 11s - 30ms/step - ia: 0.4293 - loss: 0.8427 - mae: 0.6716 - rmse: 0.8868 - smape: 1.1837 - val_ia: 0.2428 - val_loss: 0.4745 - val_mae: 0.5018 - val_rmse: 0.5631 - val_smape: 1.1866

Epoch 4/128                                                                      

385/385 - 13s - 35ms/step - ia: 0.4488 - loss: 0.8142 - mae: 0.6567 - rmse: 0.8739 - sma

...

Epoch 1/128                                                                      

385/385 - 13s - 33ms/step - ia: 0.2689 - loss: 1.2432 - mae: 0.8347 - rmse: 1.0798 - smape: 1.4768 - val_ia: 0.2394 - val_loss: 0.5285 - val_mae: 0.5616 - val_rmse: 0.6168 - val_smape: 1.6447

Epoch 2/128                                                                      

385/385 - 9s - 22ms/step - ia: 0.2659 - loss: 1.1863 - mae: 0.8152 - rmse: 1.0550 - smape: 1.4796 - val_ia: 0.2558 - val_loss: 0.4847 - val_mae: 0.5132 - val_rmse: 0.5688 - val_smape: 1.2623

Epoch 3/128                                                                      

385/385 - 6s - 14ms/step - ia: 0.3083 - loss: 1.0778 - mae: 0.7771 - rmse: 1.0033 - smape: 1.4065 - val_ia: 0.2537 - val_loss: 0.4528 - val_mae: 0.5016 - val_rmse: 0.5562 - val_smape: 1.2693

Epoch 4/128                                                                      

385/385 - 5s - 14ms/step - ia: 0.3583 - loss: 1.0059 - mae: 0.7505 - rmse: 0.9747 - smape: 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



770/770 - 14s - 18ms/step - ia: 0.2714 - loss: 1.2039 - mae: 0.7714 - rmse: 1.0106 - smape: 1.4620 - val_ia: 0.2104 - val_loss: 0.5278 - val_mae: 0.5352 - val_rmse: 0.5729 - val_smape: 1.3384

Epoch 2/128                                                                       

770/770 - 11s - 14ms/step - ia: 0.2665 - loss: 1.1960 - mae: 0.7675 - rmse: 1.0069 - smape: 1.4550 - val_ia: 0.2101 - val_loss: 0.5275 - val_mae: 0.5357 - val_rmse: 0.5734 - val_smape: 1.3446

Epoch 3/128                                                                       

770/770 - 5s - 6ms/step - ia: 0.2642 - loss: 1.2000 - mae: 0.7724 - rmse: 1.0077 - smape: 1.4730 - val_ia: 0.2098 - val_loss: 0.5273 - val_mae: 0.5363 - val_rmse: 0.5740 - val_smape: 1.3516

Epoch 4/128                                                                       

770/770 - 6s - 7ms/step - ia: 0.2609 - loss: 1.1990 - mae: 0.7735 - rmse: 1.0083 - smape: 1.4857 - val_ia: 0.2096 - val_loss: 0.5271 - val_mae: 0.5368 - val_rmse: 0.5745 -

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                       

49/49 - 17s - 339ms/step - ia: 0.2352 - loss: 1.1199 - mae: 0.7935 - rmse: 1.0496 - smape: 1.5011 - val_ia: 0.2836 - val_loss: 0.4699 - val_mae: 0.5126 - val_rmse: 0.6367 - val_smape: 1.3220

Epoch 2/128                                                                       

49/49 - 1s - 30ms/step - ia: 0.3229 - loss: 1.0129 - mae: 0.7540 - rmse: 1.0021 - smape: 1.3762 - val_ia: 0.3029 - val_loss: 0.4505 - val_mae: 0.4883 - val_rmse: 0.6185 - val_smape: 1.1702

Epoch 3/128                                                                       

49/49 - 2s - 32ms/step - ia: 0.3679 - loss: 0.9818 - mae: 0.7365 - rmse: 0.9822 - smape: 1.2994 - val_ia: 0.3031 - val_loss: 0.4448 - val_mae: 0.5080 - val_rmse: 0.6314 - val_smape: 1.3007

Epoch 4/128                                                                       

49/49 - 2s - 35ms/step - ia: 0.3912 - loss: 0.9530 - mae: 0.7283 - rmse: 0.9806 - smape: 1.2

In [16]:
print(best)

{'activation': 3, 'batch': 4, 'dropout': 0.4, 'layers': 1.0, 'learning_rate': 0.009726326397617627, 'units': 3}


In [17]:
#{'activation': 1, 'batch': 4, 'dropout': 0.2, 'layers': 1.0, 'learning_rate': 0.0002707756079796208, 'units': 4}